# Experimentation notebook for hybrid CNNs

## Dependencies

In [1]:
import os, sys
from pathlib import Path

import numpy as np
import scipy as sp
import scipy.sparse as sps
import torch
import neuropythy as ny

import matplotlib as mpl
import matplotlib.pyplot as plt
import ipyvolume as ipv

sys.path.append('../src')
import visual_autolabel as val

## Configuration

In [2]:
dataset2D_cache_path = '/data/visual-autolabel/datasets/HCP'

val.image.dataset3D_cache_path = '/data/visual-autolabel/volumetric/data'
val.image.noddi_data_path = '/data/visual-autolabel/volumetric/NODDI'

inputs3D = ('graymask', 'T1', 'T2')
inputs2D = ('curvature', 'convexity', 'thickness')
outputs = ('V1', 'V2', 'V3')
batch_size = 5
lr = 0.0075
gamma = 0.95
num_epochs = 30
zoom = 1/2
subindex = (slice(2,-2), slice(8, 256+8), slice(2,-2))
dtype = torch.float32
device = 'cpu'
shuffle = True

## Setting up Training Loop

In [3]:
# Training and validation subjects
trn_sids = [100610, 118225, 140117, 158136, 197348, 214524, 346137, 412528,
            573249, 724446, 905147, 102311, 159239, 173334, 221319, 352738,
            429040, 725751, 826353, 910241, 102816, 145834, 162935, 175237,
            199655, 233326, 436845, 732243, 926862, 104416, 128935, 146129,
            164131, 200210, 365343, 463040, 751550, 859671, 927359, 105923,
            130114, 146432, 164636, 187345, 200311, 467351, 617748, 757764,
            942658, 108323, 130518, 165436, 191033, 200614, 249947, 381038,
            525541, 627549, 109123, 131217, 146937, 167036, 177746, 191336,
            201515, 385046, 536647, 638049, 770352, 872764, 111312, 167440,
            178142, 191841, 203418, 257845, 541943, 771354, 878776, 958976,
            111514, 132118, 169040, 178243, 393247, 547046, 654552, 878877,
            966975, 114823, 155938, 205220, 283543, 395756, 550439, 671855,
            783462, 898176, 971160, 156334, 180533, 193845, 318637, 397760,
            552241, 680957, 899885, 973770, 115825, 135124, 157336, 169747,
            181232, 401422, 562345, 690152, 814649, 901139, 995174, 116726,
            137128, 158035, 181636, 196144, 330324, 406836, 572045, 818859]
val_sids = [765864, 209228, 134829, 585256, 901442, 169444, 380036, 389357,
            581450, 198653, 115017, 782561, 176542, 246133, 185442, 601127,
            204521, 195041, 182739, 212419, 263436, 320826, 825048, 192641,
            360030, 177140, 146735, 126426, 789373, 871762, 172130, 171633]

In [4]:
trn_dataset3D = val.image._data3D.HCPVolumeDataset(
    sids=trn_sids,
    inputs=inputs3D,
    outputs=outputs,
    cache_path=val.image.dataset3D_cache_path,
    dtype=dtype,
    device=device,
    mkdir_mode=0o775,
    subindex=subindex,
    zoom=zoom)

In [5]:
trn_dataset2D = val.benson2025.hcp.HCPDataset(
    inputs2D, outputs,
    sids=trn_sids)

In [18]:
class VolumeToFlatImageDataset(torch.utils.data.Dataset):
    def __init__(self,
                 sids,
                 cache_path='/data/visual-autolabel/volumetric/data/affines'):
        self.cache_path = Path(cache_path)
        self.sids = sids
        self.data = {}
    def __len__(self):
        return len(self.sids)
    def __getitem__(self, k):
        sid = self.sids[k]
        if sid in self.data:
            return self.data[sid]
        matrix = torch.load(f"{self.cache_path}/{sid}.pt")
        import scipy.sparse as sps
        (row, col, val) = sps.find(matrix)
        matrix = torch.sparse_csr_tensor(
            torch.as_tensor(row), torch.as_tensor(col), 
            torch.as_tensor(val),
            matrix.shape)
        matrix = torch.tensor(matrix, dtype=torch.float32)
        self.data[sid] = matrix
        return matrix

In [19]:
trn_transforms = VolumeToFlatImageDataset(trn_sids)

In [20]:
model = val.image.UNet(
    len(inputs3D),
    len(inputs3D),
    len(inputs2D),
    len(outputs))

In [21]:
k = 0

(features3D, _) = trn_dataset3D[k]
(features2D, labels) = trn_dataset2D[k]
transform = trn_transforms[k]

outputs = model(features3D[None,...], features2D[None,...], transform)

/tmp/ipykernel_4141/3295770229.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  matrix = torch.tensor(matrix, dtype=torch.float32)


RuntimeError: shape '[1, 3, 128, 256]' is invalid for input of size 1572864

In [12]:
# Make 3D Dataloaders:
(trn_loader, val_loader) = val.image._data3D.make_dataloaders(
    inputs=inputs,
    outputs=outputs, 
    zoom=zoom,
    dtype=dtype,
    device=device,
    batch_size=batch_size,
    shuffle=shuffle,
    partition=(trn_sids, val_sids))
dataloaders = {'trn': trn_loader, 'val': val_loader}




TypeError: HybridUNet.__init__() missing 2 required positional arguments: 'feature_count_2D' and 'segment_count'

In [ ]:
# Make the model.
model = val.image.UNet(len(inputs), len(outputs))

In [ ]:
# We use the Adam optimizer.
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=lr)
# We use the StepLR scheduler.
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=2,
    gamma=gamma)